In [1]:
import pandas as pd
cocoa = pd.read_csv("data/clean/cocoa_flows_clean.csv")
palm  = pd.read_csv("data/clean/palm_oil_flows_clean.csv")
iso   = pd.read_csv("data/clean/country_iso3.csv")

In [2]:
cocoa_h = cocoa.rename(columns={
    "department_of_production": "origin_region",
    "logistics_hub_cooperative": "source_unit",
    "trader_group": "trader",
    "country_of_destination": "destination_country",
    "is_unknown_origin": "is_unknown_source"})
cocoa_h["origin_country"] = "COTE D'IVOIRE"

palm_h = palm.rename(columns={
    "province_of_production": "origin_region",
    "mill_group": "source_unit",
    "exporter": "trader",
    "country_of_first_import": "destination_country",
    "is_unknown_mill": "is_unknown_source"})
palm_h["origin_country"] = "INDONESIA"

In [3]:
flows = pd.concat([cocoa_h, palm_h], ignore_index=True)
flows = flows[["year", "commodity", "origin_country", "origin_region", "source_unit", "trader",
               "destination_country", "volume", "is_unknown_source", "is_eu", "eu_volume", "unknown_volume"]]

print(len(flows), len(cocoa) + len(palm))        # these must be equal
print(flows.groupby(["commodity", "year"])["volume"].sum().round(0))
print(flows.isna().sum())

270125 270125
commodity  year
Cocoa      2020     1904048.0
           2021     1979405.0
           2022     1886302.0
           2023     1765143.0
           2024     1536045.0
Palm oil   2020    44673368.0
           2021    45029271.0
           2022    46744330.0
           2023    46980151.0
           2024    47127865.0
Name: volume, dtype: float64
year                   0
commodity              0
origin_country         0
origin_region          0
source_unit            0
trader                 0
destination_country    0
volume                 0
is_unknown_source      0
is_eu                  0
eu_volume              0
unknown_volume         0
dtype: int64


In [4]:
iso_map = dict(zip(iso["country_name"], iso["iso3"]))
flows["destination_iso3"] = flows["destination_country"].map(iso_map)
flows["origin_iso3"]      = flows["origin_country"].map(iso_map)
print("unmapped destinations:", flows["destination_iso3"].isna().sum())   # must be 0

unmapped destinations: 0


In [5]:
flows.to_csv("data/clean/trase_flows.csv", index=False)

In [ ]:
cd = pd.read_csv("data/clean/cocoa_territorial_deforestation_clean.csv").rename(columns={
    "department": "region", "region_parent": "parent_region"})
cd["commodity"] = "Cocoa";    cd["country"] = "COTE D'IVOIRE"

pdf = pd.read_csv("data/clean/palm_territorial_deforestation_clean.csv").rename(columns={
    "province": "region", "island_group": "parent_region"})
pdf["commodity"] = "Palm oil"; pdf["country"] = "INDONESIA"

defor = pd.concat([cd, pdf], ignore_index=True)[
    ["year", "commodity", "country", "region", "region_trase_id", "parent_region", "deforestation_ha"]]
defor["country_iso3"] = defor["country"].map(dict(zip(iso["country_name"], iso["iso3"])))

print(len(defor), len(cd) + len(pdf))        # 540 + 190 = 721? check: must be equal
print(defor["country_iso3"].isna().sum())    # must be 0
defor.to_csv("data/clean/territorial_deforestation.csv", index=False)

730 730
0
